In [ ]:
#réinitialise module

%load_ext autoreload
%autoreload 2


# les différentes importations 

from IPython.display import IFrame

import random
from datetime import datetime

import sys

sys.path.append('..')
import bs4
import gtfs_kit as gk
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString
from IPython.display import HTML
import folium
from folium import plugins
import numpy as np
import branca.colormap as cm

from src.utils import (
    charger_gtfs,
    longueur_lignes,
    km_par_ligne_jour,
    km_par_ligne_plage,
    obtenir_service_ids_pour_date,
    exporter_df_to_csv,
    exporter_geojson,
    exporter_gdf_to_csv,
    
)
from src.info_reseau import dates_service, formater_date_fr, date_str, longueur_par_lignes, nom_reseau_str, chemin_logo, recuperer_logo_reseau, nom_reseau 

from src.arrets import calculer_indicateurs_arrets, afficher_statistiques
from src.cartographie import creer_carte_troncons, create_carte_arrets 
from src.create_troncons_uniques import creer_troncons_uniques
from src.indicateurs_troncons import compute_indicateurs_troncons

from src.export_html import (
    exporter_tableau_lignes_html,
    exporter_camembert_html,
    exporter_statistiques_html,
)



In [ ]:
#les chemins 


# Chemin vers le zip GTFS
BASE_DIR = os.getcwd()  # Remonte d'un niveau depuis scripts/

print(BASE_DIR)

# Choisir le jeu de données GTFS en zip, site de référence 
GTFS_ZIP_PATH = os.path.join(BASE_DIR,"data", "TAM_MMM_GTFS.zip")
OUTPUT_HTML_PATH = os.path.join(BASE_DIR, "output")

print(GTFS_ZIP_PATH)
print(OUTPUT_HTML_PATH)

In [ ]:

#charge GTFS en feed, longueurs lignes et nom du réseau  


    # La librairie gtfs_kit est utilisée pour charger le GTFS
feed = charger_gtfs(GTFS_ZIP_PATH)

print(type(feed))  # Vérification du type de l'objet feed

# Calcul de la longueur des shapes une seule fois, en dehors de la boucle

longueur_par_lignes=longueur_lignes(feed)

print(longueur_par_lignes)

    #cherche nom réseau 
    
# On appelle les fonctions via le module (plutôt que les noms importés
# nom_reseau_str/chemin_logo) pour ne jamais les écraser avec leur résultat :
# sinon, réexécuter cette cellule une deuxième fois lève
# "TypeError: 'str' object is not callable".
import src.info_reseau as _info_reseau

nom_reseau_str = _info_reseau.nom_reseau_str(feed)

nb_agences = len(feed.agency)
if nb_agences > 3:
    print(f"⚠ Ce GTFS regroupe {nb_agences} agences : ce que l'app ne peut pas gérer. Charger un GTFS urbain uniquement. ")


#cherche nom réseau 
chemin_logo = _info_reseau.chemin_logo(feed)

print(nom_reseau_str)

In [ ]:

# définition plage temporelle et défintion date_JOB 

dates_service, date_debut , date_fin , date_JOB = dates_service(feed)

print(dates_service, date_debut, date_fin, date_JOB)

date_service_str, date_JOB_text = date_str(date_debut, date_fin, date_JOB)

print(date_service_str)
print(date_JOB)





In [ ]:

# Fonction pour calculer le total des kilomètres parcourus par ligne pour une journée donnée

total_vkm_per_plage=km_par_ligne_plage(dates_service,feed)

output_html_tableau=os.path.join(OUTPUT_HTML_PATH, f"tableau_ligne_plage_{nom_reseau_str}.html")

exporter_tableau_lignes_html(
    nom_reseau_str,
    date_service_str,
    feed,
    output_html_tableau,
    total_vk_plage=None,
)

HTML(filename=output_html_tableau)



In [ ]:

#export camembert avec répartition vk par mode par an 

output_html_camembert=os.path.join(OUTPUT_HTML_PATH, f"camembert_ligne_an_{nom_reseau_str}.html")

exporter_camembert_html(nom_reseau_str,date_service_str,total_vkm_per_plage, output_html_camembert)

HTML(filename=output_html_camembert)


In [ ]:
# calcul indicateurs par arrêts 

indicateurs=calculer_indicateurs_arrets(feed,date_JOB)

# Il est possible d'exporter les résultats en csv
exporter_df_to_csv(
    indicateurs,
    f"output/indicateurs_arrets_{date_JOB}_{nom_reseau_str}.csv"
)


In [ ]:
# statistique du réseau vision arrêts

output_html_statistiques=os.path.join(OUTPUT_HTML_PATH, f"statistiques_réseau_{nom_reseau_str}.html")

afficher_statistiques(indicateurs)

#exporter_statistiques_html(indicateurs, date_JOB_text, output_html_statistiques, nom_reseau_str)

exporter_statistiques_html(indicateurs, date_service_str, date_JOB_text, output_html_statistiques, nom_reseau_str)

HTML(filename=output_html_statistiques)



In [ ]:
# créer la carte des arrêts 

output_html_arret=os.path.join(OUTPUT_HTML_PATH, f"stop_maps_{nom_reseau_str}.html")

carte_arrets = create_carte_arrets(indicateurs, nom_reseau_str, date_service_str, date_JOB, GTFS_ZIP_PATH, output_html_arret, chemin_logo)


carte_arrets

In [ ]:
# Tronçons de bus
troncons_bus = creer_troncons_uniques(feed, route_type=3)
    
# Tronçons de tram
troncons_tram = creer_troncons_uniques(feed, route_type=0)

# Tronçons de metro 
troncons_metro = creer_troncons_uniques(feed, route_type=1)

# Tronçons de trolley 
troncons_trolley = creer_troncons_uniques(feed, route_type=11)

# Tronçons de ferry 
troncons_ferry = creer_troncons_uniques(feed, route_type=4)


# Il est possible d'exporter les résultats au format csv (sans géométrie, ou en geojson)
# exporter_gdf_to_csv(troncons_bus, 'output/troncons_uniques_bus.csv')
# exporter_geojson(troncons_bus, 'output/troncons_uniques_bus.geojson')

# exporter_gdf_to_csv(troncons_tram, 'output/troncons_uniques_tram.csv')
# exporter_geojson(troncons_tram, 'output/troncons_uniques_tram.geojson')

In [ ]:
# Les indicateurs sont calculés sous la forme d'un GeoDataframe par mode de transport


active_service_ids = obtenir_service_ids_pour_date(feed, date_JOB)


indicateurs_bus, indicateurs_tram, indicateurs_metro, indicateurs_trolley, indicateurs_ferry = compute_indicateurs_troncons(
    feed,  # le feed GTFS chargé plus haut
    active_service_ids,  # prise en compte uniquement des services actifs ce jour
    troncons_bus,  # Geodataframe des tronçons de bus
    troncons_tram,  # Geodataframe des tronçons de tram
    troncons_metro, # Geodataframe des tronçons de metro
    troncons_trolley, # Geodataframe des tronçons de trolley
    troncons_ferry # Geodataframe des tronçonsde de ferry
)

# Il est possible d'exporter les résultats au format csv (sans géométrie, ou en geojson)
# exporter_gdf_to_csv(indicateurs_bus, f'output/indicateurs_troncons_bus_{DATE_ANALYSE}.csv')
# exporter_gdf_to_csv(indicateurs_tram, f'output/indicateurs_troncons_tram_{DATE_ANALYSE}.csv')

# exporter_geojson(indicateurs_bus, f'output/indicateurs_troncons_bus_{DATE_ANALYSE}.geojson')
# exporter_geojson(indicateurs_tram, f'output/indicateurs_troncons_tram_{DATE_ANALYSE}.geojson')

In [ ]:

# Créer la carte des tronçons

output_html_troncon=os.path.join(OUTPUT_HTML_PATH, f"troncon_maps_{nom_reseau_str}.html")

carte_troncons = creer_carte_troncons(
    indicateurs_bus,
    indicateurs_tram,
    indicateurs_metro,
    indicateurs_trolley,
    indicateurs_ferry,
    output_html_troncon,
    date_service_str,
    colonne_frequence='nombre_passages',
    nom_reseau_str=nom_reseau_str,
    chemin_logo=chemin_logo,
)

# Afficher la carte
carte_troncons

